In [3]:
# ===============================
# 1️⃣ Imports
# ===============================
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

c:\Users\USER\miniconda3\envs\env_RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# ===============================
# 2️⃣ OpenRouter Config
# ===============================
OPENROUTER_API_KEY = "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"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


In [5]:
# ===============================
# 3️⃣ Load PDF
# ===============================
pdf_path = "TST.2025.9010097.pdf"  # Replace with your PDF file
loader = PyPDFLoader(pdf_path)
documents = loader.load()
print(f"Loaded {len(documents)} pages from PDF")

Loaded 43 pages from PDF


In [6]:
# ===============================
# 4️⃣ Split PDF into Chunks
# ===============================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)
print(f"Split into {len(docs)} chunks")

Split into 448 chunks


In [7]:
# ===============================
# 5️⃣ Embeddings
# ===============================
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL
)

In [8]:
# ===============================
# 6️⃣ Vector Store (FAISS)
# ===============================
vector_db = FAISS.from_documents(docs, embeddings)
retriever = vector_db.as_retriever()


In [9]:
# ===============================
# 7️⃣ LLM (OpenRouter GPT)
# ===============================
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0
)

In [10]:
# ===============================
# 8️⃣ Prompt Template
# ===============================
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question:
{question}
""")

In [11]:
# ===============================
# 9️⃣ LCEL RAG Chain
# ===============================
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [14]:
# ===============================
# 🔟 Ask a Question
# ===============================
query = "What is the main theme of the document?"
answer = rag_chain.invoke(query)

print("\nAnswer:\n", answer)



Answer:
 The main theme of the document revolves around the advancements in large language models (LLMs), particularly focusing on the automation of the labeling process, ethical and legal considerations in technology deployment, and the role of human annotators in enhancing language processing capabilities.
